In [ ]:
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
uploaded = files.upload()

Saving healthcare_dataset_dirty-1.csv to healthcare_dataset_dirty-1.csv


In [ ]:
df = pd.read_csv("healthcare_dataset_dirty-1.csv")

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 54859
Columns: 24


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54859 entries, 0 to 55499
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Name                54859 non-null  object        
 1   Age                 54859 non-null  Int64         
 2   Gender              54859 non-null  object        
 3   Blood Type          54859 non-null  object        
 4   Medical Condition   54859 non-null  object        
 5   Date of Admission   54859 non-null  datetime64[ns]
 6   Doctor              54859 non-null  object        
 7   Hospital            54859 non-null  object        
 8   Insurance Provider  54859 non-null  object        
 9   Billing Amount      54859 non-null  float64       
 10  Room Number         54859 non-null  int64         
 11  Admission Type      54859 non-null  object        
 12  Discharge Date      54859 non-null  datetime64[ns]
 13  Medication          54859 non-null  object        


In [ ]:
display(df.describe(include="all"))

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
count,55504,55504.000000,55504,55504,55504,55504,55504,55504,55504,55504.000000,55505.000000,55505,55505,55505,55505
unique,49991,NaN,3,9,7,1828,40340,39875,6,NaN,NaN,4,1857,5,3
top,adrIENNE bEll,NaN,Male,A-,Arthritis,2024-03-16,Michael Smith,LLC Smith,Cigna,NaN,NaN,Elective,2020-03-15,Lipitor,Abnormal
freq,3,NaN,27775,6970,9308,50,27,44,11249,NaN,NaN,18656,53,11140,18629
mean,NaN,51.539367,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25539.436509,301.138438,NaN,NaN,NaN,NaN
std,NaN,19.602490,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14211.191930,115.242415,NaN,NaN,NaN,NaN
min,NaN,13.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2008.492140,101.000000,NaN,NaN,NaN,NaN
25%,NaN,35.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13242.605454,202.000000,NaN,NaN,NaN,NaN
50%,NaN,52.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,25538.694782,302.000000,NaN,NaN,NaN,NaN
75%,NaN,68.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37820.508436,401.000000,NaN,NaN,NaN,NaN


Data Cleaning

In [ ]:
missing_count = df.isnull().sum() #Checking Missing Values
display(missing_count[missing_count > 0])

,0
Name,1
Age,1
Gender,1
Blood Type,1
Medical Condition,1
Date of Admission,1
Doctor,1
Hospital,1
Insurance Provider,1
Billing Amount,1


In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100 # Check missing-value percentage
display(missing_percentage[missing_percentage > 0])

,0
Name,0.001802
Age,0.001802
Gender,0.001802
Blood Type,0.001802
Medical Condition,0.001802
Date of Admission,0.001802
Doctor,0.001802
Hospital,0.001802
Insurance Provider,0.001802
Billing Amount,0.001802


In [ ]:
# Total missing values in entire dataset
total_missing = df.isnull().sum().sum()
# Total cells in entire dataset
total_cells = df.shape[0] * df.shape[1]
# Overall missing value percentage
overall_missing_percentage = (total_missing / total_cells) * 100
print("Total Missing Values:", total_missing)
print("Total Cells:", total_cells)
print("Overall Missing Value Percentage:",
      round(overall_missing_percentage, 4), "%") #Show total missing values

Total Missing Values: 10
Total Cells: 832575
Overall Missing Value Percentage: 0.0012 %


In [ ]:
# Check invalid negative billing amounts
negative_billing = df[df["Billing Amount"] < 0]

print("Negative billing records:", len(negative_billing))

display(
    negative_billing[
        ["Name", "Billing Amount"]
    ].head(20)
)

Negative billing records: 106


,Name,Billing Amount
132,Ashley Erickson,-502.51
799,Christopher Weiss,-1018.25
1018,Ashley Warner,-306.36
1421,Jay Galloway,-109.10
2103,Joshua Williamson,-576.73
2696,Scott Vazquez,-135.99
2855,Carol Anderson,-370.98
3772,Mr. Christopher Alvarado,-1310.27
5445,Alexandra Khan,-692.41
5708,Joseph Cox,-353.87


In [ ]:
df = df[df["Billing Amount"] >= 0].copy()

print(
    "Negative billing amounts after cleaning:",
    (df["Billing Amount"] < 0).sum()
)

Negative billing amounts after cleaning: 0


In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns # Numeric Columns

for col in numeric_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
text_columns = df.select_dtypes(include="object").columns # Text Columns

for col in text_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
print("Total missing values after cleaning:", df.isnull().sum().sum()) # After cleaning

Total missing values after cleaning: 0


In [ ]:
duplicate_count = df.duplicated().sum() #Check duplicate records
print("Duplicate records found:", duplicate_count)

Duplicate records found: 539


In [ ]:
if duplicate_count > 0: # Duplicate Records
    display(df[df.duplicated(keep=False)])
else:
    print("No duplicate records found.")

,Name,Age,Gender,Blood Type,Medical Condition,Date of Admission,Doctor,Hospital,Insurance Provider,Billing Amount,Room Number,Admission Type,Discharge Date,Medication,Test Results
0,Bobby JacksOn,30.0,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
1,LesLie TErRy,62.0,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
2,DaNnY sMitH,76.0,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
3,andrEw waTtS,28.0,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal
4,adrIENNE bEll,43.0,Female,AB+,Cancer,2022-09-19,Kathleen Hanna,White-White,Aetna,14238.317814,458,Urgent,2022-10-09,Penicillin,Abnormal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55500,Bobby JacksOn,30.0,Male,B-,Cancer,2024-01-31,Matthew Smith,Sons and Miller,Blue Cross,18856.281306,328,Urgent,2024-02-02,Paracetamol,Normal
55501,LesLie TErRy,62.0,Male,A+,Obesity,2019-08-20,Samantha Davies,Kim Inc,Medicare,33643.327287,265,Emergency,2019-08-26,Ibuprofen,Inconclusive
55502,DaNnY sMitH,76.0,Female,A-,Obesity,2022-09-22,Tiffany Mitchell,Cook PLC,Aetna,27955.096079,205,Emergency,2022-10-07,Aspirin,Normal
55503,andrEw waTtS,28.0,Female,O+,Diabetes,2020-11-18,Kevin Wells,"Hernandez Rogers and Vang,",Medicare,37909.782410,450,Elective,2020-12-18,Ibuprofen,Abnormal


In [ ]:
if duplicate_count > 0: # Removing Duplicates
    df = df.drop_duplicates()
    print("Duplicate records removed.")
else:
    print("No duplicates to remove.")

Duplicate records removed.


In [ ]:
print("Duplicate records after cleaning:", df.duplicated().sum()) # Verifying Duplicates

Duplicate records after cleaning: 0


In [ ]:
df = df.copy()
text_columns = df.select_dtypes(include="object").columns # Check leading/trailing spaces
for col in text_columns:
    space_rows = df[df[col].notna() & (df[col] != df[col].str.strip())]
    if len(space_rows) > 0:
        print(f"\nColumn: {col}")
        print("Rows containing leading/trailing spaces:", len(space_rows))
        display(space_rows[[col]].head())

In [ ]:
for col in text_columns: # Removing Spaces
    df[col] = df[col].str.strip()

In [ ]:
remaining_spaces = 0 # Verifying spaces
for col in text_columns:
    remaining_spaces += (
        df[col].notna() & (df[col] != df[col].str.strip())
    ).sum()
print("Remaining leading/trailing spaces:", remaining_spaces)

Remaining leading/trailing spaces: 0


In [ ]:
# Check inconsistent capitalization in all text columns

text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(20))

===== BEFORE STANDARDIZATION =====

--- Name ---
Name
Michael Williams     24
Michael Smith        23
Robert Smith         21
James Brown          19
James Smith          18
John Johnson         16
John Smith           16
James Williams       16
Kimberly Smith       16
James Garcia         16
Jennifer Jones       15
David Johnson        15
Matthew Smith        15
Jennifer Smith       14
Christopher Smith    14
Thomas Smith         14
Matthew Jones        14
William Smith        14
Robert Williams      14
Michael Jones        14
Name: count, dtype: int64

--- Gender ---
Gender
Male      27496
Female    27469
mALE          1
Name: count, dtype: int64

--- Blood Type ---
Blood Type
A-     6899
A+     6896
B+     6885
AB+    6881
AB-    6874
B-     6872
O+     6854
O-     6804
o+        1
Name: count, dtype: int64

--- Medical Condition ---
Medical Condition
Arthritis       9219
Diabetes        9216
Hypertension    9151
Obesity         9146
Cancer          9139
Asthma          9094
aSTHMA 

In [ ]:

# Standardize capitalization
for col in text_columns:
    df[col] = df[col].str.strip()
    # General text standardization
    if col != "Blood Type":
        df[col] = df[col].str.title()
# Blood Type needs uppercase format
if "Blood Type" in df.columns:
    df["Blood Type"] = df["Blood Type"].str.upper()
# Insurance Provider correction
if "Insurance Provider" in df.columns:
    df["Insurance Provider"] = (
        df["Insurance Provider"]
        .str.lower()
        .replace({
            "cigna": "Cigna",
            "medicare": "Medicare",
            "unitedhealthcare": "UnitedHealthcare",
            "blue cross": "Blue Cross",
            "aetna": "Aetna"
        })
    )

print("Standardization applied successfully.")

Standardization applied successfully.


In [ ]:
for col in text_columns:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(20))

===== AFTER STANDARDIZATION =====

--- Name ---
Name
Michael Williams     24
Michael Smith        23
Robert Smith         21
James Brown          19
James Smith          18
John Johnson         16
John Smith           16
James Williams       16
Kimberly Smith       16
James Garcia         16
Jennifer Jones       15
David Johnson        15
Matthew Smith        15
Jennifer Smith       14
Christopher Smith    14
Thomas Smith         14
Matthew Jones        14
William Smith        14
Robert Williams      14
Michael Jones        14
Name: count, dtype: int64

--- Gender ---
Gender
Male      27497
Female    27469
Name: count, dtype: int64

--- Blood Type ---
Blood Type
A-     6899
A+     6896
B+     6885
AB+    6881
AB-    6874
B-     6872
O+     6855
O-     6804
Name: count, dtype: int64

--- Medical Condition ---
Medical Condition
Arthritis       9219
Diabetes        9216
Hypertension    9151
Obesity         9146
Cancer          9139
Asthma          9095
Name: count, dtype: int64

--- Date 

Transformations


In [ ]:
print(df.dtypes)# checking data types

Name                          object
Age                            Int64
Gender                        object
Blood Type                    object
Medical Condition             object
Date of Admission     datetime64[ns]
Doctor                        object
Hospital                      object
Insurance Provider            object
Billing Amount               float64
Room Number                    int64
Admission Type                object
Discharge Date        datetime64[ns]
Medication                    object
Test Results                  object
First Name                    object
Last Name                     object
Length of Stay                 int64
Age Group                   category
Billing Category            category
Admission Month               object
Admission Year                 int32
Stay Category               category
dtype: object


In [ ]:
df["Date of Admission"] = pd.to_datetime(
    df["Date of Admission"],
    errors="coerce"
)

df["Discharge Date"] = pd.to_datetime(
    df["Discharge Date"],
    errors="coerce"
)

In [ ]:
df["Age"] = pd.to_numeric(
    df["Age"],
    errors="coerce"
).astype("Int64")

In [ ]:
print("Data Types After Correction:") #After Transformation
display(df.dtypes)

Data Types After Correction:


,0
Name,object
Age,Int64
Gender,object
Blood Type,object
Medical Condition,object
Date of Admission,datetime64[ns]
Doctor,object
Hospital,object
Insurance Provider,object
Billing Amount,float64


In [ ]:
df["Billing Amount"] = df["Billing Amount"].round(2) # Billing amount rounded to 2 decimals
display(df[["Billing Amount"]].head())

,Billing Amount
0,18856.28
1,33643.33
2,27955.10
3,37909.78
4,14238.32


In [ ]:
df["First Name"] = df["Name"].str.split().str[0] # Splitting First and last names
df["Last Name"] = df["Name"].str.split().str[-1]
display(
    df[["Name", "First Name", "Last Name"]].head()
)

,Name,First Name,Last Name
0,Bobby Jackson,Bobby,Jackson
1,Leslie Terry,Leslie,Terry
2,Danny Smith,Danny,Smith
3,Andrew Watts,Andrew,Watts
4,Adrienne Bell,Adrienne,Bell


In [ ]:
df["Length of Stay"] = (
    df["Discharge Date"] - df["Date of Admission"]
).dt.days

In [ ]:
display(            # Creating length of stay
    df[
        [
            "Date of Admission",
            "Discharge Date",
            "Length of Stay"
        ]
    ].head()
)

,Date of Admission,Discharge Date,Length of Stay
0,2024-01-31,2024-02-02,2
1,2019-08-20,2019-08-26,6
2,2022-09-22,2022-10-07,15
3,2020-11-18,2020-12-18,30
4,2022-09-19,2022-10-09,20


In [ ]:
# Dividing the age Group
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 120],
    labels=[
        "Child",
        "Teenager",
        "Young Adult",
        "Adult",
        "Senior"
    ],
    include_lowest=True
)
display(
    df[["Age", "Age Group"]].head(10)
)

,Age,Age Group
0,30,Young Adult
1,62,Senior
2,76,Senior
3,28,Young Adult
4,43,Adult
5,36,Adult
6,21,Young Adult
7,20,Young Adult
8,82,Senior
9,58,Adult


In [ ]:
# Creating Billing Category
df["Billing Category"] = pd.cut(
    df["Billing Amount"],
    bins=[0, 10000, 25000, 50000, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ]
)
display(
    df[
        ["Billing Amount", "Billing Category"]
    ].head(10)
)

,Billing Amount,Billing Category
0,18856.28,Medium
1,33643.33,High
2,27955.10,High
3,37909.78,High
4,14238.32,Medium
5,48145.11,High
6,19580.87,Medium
7,45820.46,High
8,50119.22,Very High
9,19784.63,Medium


In [ ]:
# Extracting month and year
df["Admission Month"] = df["Date of Admission"].dt.month_name()
df["Admission Year"] = df["Date of Admission"].dt.year
display(
    df[
        ["Date of Admission", "Admission Month", "Admission Year"]
    ].head(10)
)


,Date of Admission,Admission Month,Admission Year
0,2024-01-31,January,2024
1,2019-08-20,August,2019
2,2022-09-22,September,2022
3,2020-11-18,November,2020
4,2022-09-19,September,2022
5,2023-12-20,December,2023
6,2020-11-03,November,2020
7,2021-12-28,December,2021
8,2020-07-01,July,2020
9,2021-05-23,May,2021


In [ ]:
# staying category based on how many days they stayed
df["Stay Category"] = pd.cut(
    df["Length of Stay"],
    bins=[-1, 7, 14, 30, float("inf")],
    labels=[
        "Short Stay",
        "Medium Stay",
        "Long Stay",
        "Very Long Stay"
    ]
)
display(
    df[
        ["Length of Stay", "Stay Category"]
    ].head(10)
)


,Length of Stay,Stay Category
0,2,Short Stay
1,6,Short Stay
2,15,Long Stay
3,30,Long Stay
4,20,Long Stay
5,4,Short Stay
6,12,Medium Stay
7,10,Medium Stay
8,13,Medium Stay
9,30,Long Stay


In [ ]:
print("Total Patients:",
      len(df))
print("Total Billing Amount:",
      round(df["Billing Amount"].sum(), 2))
print("Average Billing Amount:",
      round(df["Billing Amount"].mean(), 2))
print("Average Length of Stay:",
      round(df["Length of Stay"].mean(), 2),
      "days")
print("Total Doctors:",
      df["Doctor"].nunique())
print("Total Hospitals:",
      df["Hospital"].nunique())

Total Patients: 54859
Total Billing Amount: 1404111113.1
Average Billing Amount: 25594.91
Average Length of Stay: 15.5 days
Total Doctors: 40274
Total Hospitals: 39813


In [ ]:
if "Patient Count" in df.columns:
    df.drop(columns=["Patient Count"], inplace=True)

print("Patient Count column removed.")

Patient Count column removed.


In [ ]:
# Check final dataset
print("Final dataset shape:", df.shape)
# Save the cleaned and transformed dataset
file_name = "hospital_cleaned_transformed_dataset.csv"
df.to_csv(file_name, index=False)
print("Dataset saved successfully as:", file_name)

Final dataset shape: (54859, 23)
Dataset saved successfully as: hospital_cleaned_transformed_dataset.csv


In [ ]:
from google.colab import files

files.download("hospital_cleaned_transformed_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>